# Vaizdų generavimo programų kūrimas (OpenAI)

Didžiausias dėmesys LLM nėra tik teksto generavimui. Taip pat galite generuoti vaizdus pagal teksto aprašymus. Vaizdai kaip modališkumas yra naudingi medicinos technologijose, architektūroje, turizme, žaidimų kūrime, rinkodaroje ir dar daugiau. Šiame pamokoje dirbsime su šiuolaikiniais **GPT Image** modeliais OpenAI platformoje.

## Mokymosi tikslai

Pamokos pabaigoje galėsite:

- Paaiškinti, kas yra vaizdų generavimas ir kur jis yra naudingas.
- Suprasti `gpt-image` modelių šeimą ir kuo ji skiriasi nuo senųjų DALL·E modelių.
- Sukurti vaizdų generavimo programą ir redaguoti vaizdus.

## Kas yra vaizdų generavimas?

Vaizdų generavimo modeliai kuria vaizdus pagal teksto užklausą. Šiuolaikiniai modeliai, tokie kaip `gpt-image`, mokosi ryšio tarp teksto ir vaizdų treniruotės metu, tada iteratyviai paverčia atsitiktinį triukšmą į vaizdą, atitinkantį jūsų užklausą.

Dvi gerai žinomos vaizdų modelių šeimos yra:

- **`gpt-image` (OpenAI)** — dabartinės kartos modeliai, naudojami šioje pamokoje. Jie palaiko tekstą į vaizdą generavimą ir vaizdų redagavimą (užtapymą su kauke).
- **Midjourney** — populiarus trečiosios šalies modelis su savo paslauga ir Discord pagrindu veikiančiu darbo procesu.

> Senesni OpenAI vaizdų modeliai — **DALL·E 2** ir **DALL·E 3** — yra archyviniai. DALL·E 3 nebeprieinamas naujiems diegimams, o tokios funkcijos kaip `create_variation` buvo tik DALL·E 2. Naujiems taikymams naudokite `gpt-image` modelius.

> **Svarbu:** `gpt-image` modeliai grąžina sugeneruotą vaizdą kaip **base64** (`b64_json`), o ne kaip URL. Jūsų kodas iššifruoja base64 eilutę į baitus ir juos išsaugo — nėra vaizdo URL, kurį būtų galima atsisiųsti.


## Sukurkite savo pirmąją vaizdų generavimo programą

Taigi, ko reikia, norint sukurti vaizdų generavimo programą? Jums reikalingos šios bibliotekos:

- **python-dotenv**, labai rekomenduojama naudoti šią biblioteką, kad slaptus duomenis saugotumėte faile *.env*, atskirtame nuo kodo.
- **openai**, ši biblioteka skirta bendrauti su OpenAI API.
- **pillow**, darbui su vaizdais Python kalboje.


1. Sukurkite failą *.env* su šiuo turiniu:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Surinkite aukščiau paminėtas bibliotekas į failą, pavadintą *requirements.txt*, taip:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Tada sukurkite virtualią aplinką ir įdiekite bibliotekas:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!PASTABA]
> „Windows“ naudokite šias komandas, norėdami sukurti ir suaktyvinti savo virtualią aplinką:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Pridėkite šį kodą faile pavadinimu *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Paaiškinkime šį kodą:

- Pirmiausia importuojame mums reikalingas bibliotekas, įskaitant OpenAI biblioteką, dotenv biblioteką, base64 modulį ir Pillow biblioteką.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Po to sukuriame klientą, kuris nuskaito API raktą iš jūsų ``.env``.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Toliau generuojame paveikslėlį:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` modeliai grąžina paveikslėlį kaip **base64** eilutę `data[0].b64_json` formatu. Mes ją dekoduojame į baitus ir įrašome į failą — nėra URL, iš kurio būtų galima atsisiųsti.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Galiausiai atidarome paveikslėlį ir naudojame standartinį paveikslėlių žiūryklę jį parodyti:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Daugiau informacijos apie paveikslėlio generavimą

Pažvelkime į `images.generate` parametrus:

- **model**, yra paveikslėlio modelis, kuris bus naudojamas (pavyzdžiui, `gpt-image-2`).
- **prompt**, yra teksto užklausa, naudojama paveikslėliui sukurti. Čia ji yra „Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils“.
- **size**, yra sugeneruoto paveikslėlio dydis (pavyzdžiui, `1024x1024`, `1536x1024`, `1024x1536` arba `"auto"`).
- **n**, yra sugeneruotų paveikslėlių skaičius. Čia generuojame vieną.

> Paveikslėlių modeliai nepriima `temperature` parametro — tai yra tekstui generuoti skirtas valdymas. Norėdami gauti įvairovės, iškvieskite API dar kartą; norėdami sumažinti įvairovę, suformuluokite savo užklausą labiau specifinę.

## Papildomos paveikslėlių generavimo galimybės

Jūs jau matėte, kaip su keliais Python eilutėmis sugeneruoti paveikslėlį. `gpt-image` modeliai taip pat gali **redaguoti** esamą paveikslėlį. Pateikdami paveikslėlį, pasirenkamą **maskę** (žymi dalį, kurią reikia pakeisti) ir užklausą, galite pakeisti dalį paveikslėlio — pavyzdžiui, pridėti skrybėlę mūsų zuikučiui.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Bazinis paveikslėlis rodo tik kiškį; galutiniame paveikslėlyje kiškis jau su skrybėle.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
